# Kyber-512 KEM: Hardware NTT Accelerator Demo

Demonstrates a Kyber-style Key Encapsulation Mechanism (KEM) using the FPGA NTT accelerator
on the PYNQ-Z2. The same protocol logic runs on two backends:

- **Software** — pure-Python NTT (golden model, runs anywhere)
- **Hardware** — FPGA NTT via the C driver (`ps/ntt_driver`)

Protocol mirrors FIPS 203 ML-KEM-512: k=2 module dimension, n=256, q=3329.  
Same seed → identical shared secrets on both backends, which proves hardware correctness.

In [ ]:
%matplotlib inline
import sys, os, hashlib
import matplotlib.pyplot as plt

sys.path.insert(0, '/home/xilinx/jupyter_notebooks/kyber-ntt-fpga/ps')
sys.path.insert(0, '/home/xilinx/jupyter_notebooks/kyber-ntt-fpga/golden')
from kyber_kem import run_kem, ntt_mul_sw, ntt_mul_hw

SEED = 42
print('Setup complete')

## Step 1 — Software KEM (Python NTT)

KeyGen → Encaps → Decaps using the pure-Python golden model.

In [ ]:
print('Running software KEM...')
sw = run_kem(ntt_mul_sw, seed=SEED)
print(f"  Time:                  {sw['time_ms']:.1f} ms")
print(f"  NTT multiply calls:    {sw['mul_calls']}")
print(f"  Alice == Bob secret:   {sw['match']}")
print(f"  Shared secret:         {sw['ss_alice'].hex()[:16]}...")

## Step 2 — Hardware KEM (FPGA NTT)

Same protocol, same seed.  Each polynomial multiply now dispatches to the FPGA via `ntt_driver`.

In [ ]:
print('Running hardware KEM...')
hw = run_kem(ntt_mul_hw, seed=SEED)
print(f"  Time:                  {hw['time_ms']:.1f} ms")
print(f"  NTT multiply calls:    {hw['mul_calls']}")
print(f"  Alice == Bob secret:   {hw['match']}")
print(f"  Shared secret:         {hw['ss_alice'].hex()[:16]}...")

## Step 3 — Cross-Verification

Same seed means same random polynomials.  Hardware and software must produce identical outputs.

In [ ]:
assert sw['match'], 'Software KEM FAIL: shared secrets do not match'
assert hw['match'], 'Hardware KEM FAIL: shared secrets do not match'
assert sw['ss_alice'] == hw['ss_alice'], 'FAIL: hardware output differs from golden model'

print('Correctness checks:')
print(f"  SW Alice == SW Bob:   {sw['match']}")
print(f"  HW Alice == HW Bob:   {hw['match']}")
print(f"  SW secret == HW secret: {sw['ss_alice'] == hw['ss_alice']}")
print()
print(f"  SW: {sw['ss_alice'].hex()}")
print(f"  HW: {hw['ss_alice'].hex()}")

## Step 4 — Benchmark

In [ ]:
labels  = ['Software\n(Python NTT)', 'Hardware\n(FPGA NTT)']
times   = [sw['time_ms'], hw['time_ms']]
colors  = ['#4c72b0', '#dd8452']
speedup = sw['time_ms'] / hw['time_ms']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, times, color=colors, width=0.4)
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(times) * 0.02,
            f'{t:.1f} ms', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Total KEM time (ms)')
ax.set_title('Kyber-512 KEM: Python NTT vs FPGA NTT Accelerator')
ax.text(0.97, 0.95, f'{speedup:.1f}\u00d7 speedup', transform=ax.transAxes,
        ha='right', va='top', fontsize=13, color='green', fontweight='bold')
ax.set_ylim(0, max(times) * 1.2)
plt.tight_layout()
plt.savefig('kem_benchmark.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'FPGA is {speedup:.1f}x faster across {hw["mul_calls"]} NTT polynomial multiplications')

## Step 5 — Per-Multiply Breakdown

The KEM benchmark includes Python sampling and polynomial addition overhead on both paths.
This cell isolates the NTT multiply itself to show the raw hardware advantage.

In [ ]:
from kyber_kem import ntt_mul_sw, ntt_mul_hw
import time

a_test = list(range(256))
b_test = [(i * 7 + 3) % 3329 for i in range(256)]
REPS = 10

# Warm up the persistent hw process (first call starts it)
ntt_mul_hw(a_test, b_test)

t0 = time.time()
for _ in range(REPS):
    ntt_mul_sw(a_test, b_test)
sw_ms = (time.time() - t0) / REPS * 1000

t0 = time.time()
for _ in range(REPS):
    ntt_mul_hw(a_test, b_test)
hw_ms = (time.time() - t0) / REPS * 1000

speedup_mul = sw_ms / hw_ms

print('Per NTT multiply (n=256):')
print('  Python (ARM):   %6.1f ms' % sw_ms)
print('  FPGA (HLS IP):  %6.3f ms' % hw_ms)
print('  Speedup:        %5.0fx' % speedup_mul)
print()
print('KEM totals (%d multiplies):' % hw['mul_calls'])
print('  Software:  %.1f ms' % sw['time_ms'])
print('  Hardware:  %.1f ms' % hw['time_ms'])
print('  Speedup:   %.1fx' % (sw['time_ms'] / hw['time_ms']))

## Step 5 — Secure Message

Use the hardware-derived shared secret as an encryption key.  
Alice (Bob's encapsulation) encrypts; Bob (Alice's decapsulation) decrypts.

In [ ]:
def xor_encrypt(key: bytes, data: bytes) -> bytes:
    k = hashlib.sha256(key).digest()
    return bytes(d ^ k[i % 32] for i, d in enumerate(data))

message    = b'Hello from Kyber-accelerated FPGA!'
ciphertext = xor_encrypt(hw['ss_bob'],   message)
recovered  = xor_encrypt(hw['ss_alice'], ciphertext)

print(f"Alice encrypts:   {message.decode()!r}")
print(f"Ciphertext (hex): {ciphertext.hex()}")
print(f"Bob decrypts:     {recovered.decode()!r}")
assert recovered == message
print()
print('Secure channel established using hardware-accelerated Kyber KEM.')